In [1]:
import shutil
import os
import torch
# if mp.get_start_method(allow_none=True) != 'spawn':
#     mp.set_start_method('spawn')
from core.config import Config_2 as Config
from core.solvers import solver_entry
# %pip install pip easydict timm json_tricks xtcocotools pycocotools dict_recursive_update scikit-learn numpy

no mc
ceph can not be used
no mc
no mc
ceph can not be used


In [2]:
# set config path
# config_path = '/dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/experiments/L2_full_setting_joint_v100_32g/v100_32g_vitbase_size224.yaml'
config_path = "/dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/experiments/evaluation/attr/full_finetune/pa100k_vitbase_b64_SGD_lr1e2x05_stepLRx2_wd5e4_backboneclip_dpr03_DecoderWd0_30ep.yaml"
C = Config(config_path)

# Disable parameter for folder path
import pickle, easydict

if not os.path.exists("./temp"):
    os.makedirs("./temp")
pickle.dump(
    easydict.EasyDict({"image_name": "temp", "label": "temp", "partition": {"test": "temp"}}),
    open("./temp/temp.pkl", "wb"),
)
C.config["common"]["dataset"]["kwargs"]["task_spec"]["data_path"] = "./temp/temp.pkl"
C.config["common"]["dataset"]["kwargs"]["task_spec"]["root_path"] = "./"
C.config["expname"] = "TEMP_EXPERIMENT"

# Disable parameter for slurm
C.ginfo.neck_share_group = None
C.ginfo.group = None
C.ginfo.decoder_share_group = None

# Set pretrained model path
C.config["common"]["backbone"]["kwargs"][
    "pretrain_path"
] = "/dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/path_vitbase_size224.pth"
S = solver_entry(C)
# S.config.dataset.train=False
pseudo_dataset = S.create_dataset()
transform = pseudo_dataset.transform


[2024-09-19 08:40:33,483][           solver.py][line:  86][    INFO] auto_denan disabled!
[2024-09-19 08:40:33,486][     solver_deter.py][line:  58][    INFO] deterministic mode, seed: 42, worker_rank: True,                                   cudnn_deterministic: True


neck of task0 has been overided to {'type': 'DoNothing', 'kwargs': {}}
decoder of task0 has been overided to {'type': 'pedattr_cls_vit_A', 'kwargs': {'out_feature': 768, 'nattr': 26, 'loss_cfg': {'type': 'CEL_Sigmoid', 'kwargs': {'sample_weight': [0.04354444, 0.17997778, 0.5834, 0.4166, 0.04947778, 0.15104444, 0.10775556, 0.04191111, 0.00472222, 0.01688889, 0.03241111, 0.71171111, 0.17344444, 0.11484444, 0.006, 0.185, 0.19273333, 0.1601, 0.00952222, 0.01345556, 0.92437778, 0.06216667, 0.46044444, 0.35266667, 0.29462222, 0.35271111], 'size_average': True}}}}
dataset of task0 has been overided to {'type': 'AttrDataset', 'kwargs': {'task_spec': {'dataset': 'PA-100k', 'data_path': 'sh1424:s3://pedattr_public/PA-100k/dataset.pkl', 'root_path': 'sh1424:s3://pedattr_public/PA-100k/data/'}, 'augmentation': {'height': 256, 'width': 192, 'use_random_aug': False}}}
sampler of task0 has been overided to {'batch_size': 64, 'shuffle_strategy': 1}
sync_print: rank 0, override tensor.cuda() to preserv

In [3]:
S.config.dataset

{'type': 'AttrDataset',
 'kwargs': {'task_spec': {'dataset': 'PA-100k',
   'data_path': './temp/temp.pkl',
   'root_path': './'},
  'augmentation': {'height': 256, 'width': 192, 'use_random_aug': False},
  'ginfo': {'task_id': 0,
   'task_num': 0,
   'backbone_share_group': None,
   'task_rank': 0,
   'task_name': 'pedattr',
   'task_names': [],
   'task_weight': 1.0,
   'task_type': 'normal',
   'task_types': [],
   'task_random_seed': 0,
   'neck_share_group': None,
   'group': None,
   'decoder_share_group': None}}}

In [4]:
transform??

Signature:   transform(img)
Type:        PedAttrTestAugmentation
String form: <core.data.transforms.pedattr_transforms.PedAttrTestAugmentation object at 0x723c3bfb2240>
File:        /dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/core/data/transforms/pedattr_transforms.py
Source:     
class PedAttrTestAugmentation(object):
    def __init__(self, height, width):
        normalize = T.Normalize(mean=[0, 0, 0], std=[1, 1, 1])
        valid_transform = T.Compose([
            T.Resize((height, width)),
            T.PILToTensor(),
        ])

        self.transform = valid_transform

    def __call__(self, img):
        return self.transform(img)

In [5]:
model = S.create_model().cuda()

/dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/core/models/backbones/vitdet.py:574: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.l

load pretrain from /dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/path_vitbase_size224.pth
Missing keys: []

finish load
sync_print: rank 0, Number of conv/bn params: 0.59M
sync_print: rank 0, Number of linear params: 85.02M
[rank 0] add param pos_embed as backbone_specific
[rank 0] add param patch_embed.proj.weight as backbone_specific
[rank 0] add param patch_embed.proj.bias as backbone_specific
[rank 0] add param blocks.0.norm1.weight as backbone_specific
[rank 0] add param blocks.0.norm1.bias as backbone_specific
[rank 0] add param blocks.0.attn.qkv.weight as backbone_specific
[rank 0] add param blocks.0.attn.qkv.bias as backbone_specific
[rank 0] add param blocks.0.attn.proj.weight as backbone_specific
[rank 0] add param blocks.0.attn.proj.bias as backbone_specific
[rank 0] add param blocks.0.norm2.weight as backbone_specific
[rank 0] add param blocks.0.norm2.bias as backbone_specific
[rank 0] add param blocks.0.mlp.fc1.weight as backbone_specific
[rank 0] a

In [6]:
model.forward??

Signature: model.forward(input_var, current_step)
Docstring:
Define the computation performed at every call.

Should be overridden by all subclasses.

.. note::
    Although the recipe for forward pass needs to be defined within
    this function, one should call the :class:`Module` instance afterwards
    instead of this since the former takes care of running the
    registered hooks while the latter silently ignores them.
Source:   
    def forward(self, input_var, current_step):
        x = self.backbone_module(
            input_var
        )  # {'image': img_mask, 'label': target_mask, 'filename': img_name, 'backbone_output':xxx}
        x = self.neck_module(x)
        decoder_feature = self.decoder_module(x)
        return decoder_feature
File:      /dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/core/models/model_entry.py
Type:      method

In [7]:
model.backbone_module

ViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
     

In [14]:
temp_tensor = torch.rand(32, 3, 224, 224).cuda()
input_var = {"image": temp_tensor, "backbone_output" : True}
res = model(input_var=input_var, current_step=None)
res.keys()

tensor([[-4.3183e-01, -9.6947e-01, -1.8320e-04,  4.2523e-01,  5.5378e-01,
          4.1768e-01, -6.7516e-01,  1.9017e-01, -4.7146e-01,  3.6966e-02,
         -7.6060e-01, -8.5254e-01,  1.0514e+00, -4.9012e-01, -5.5175e-01,
          3.9320e-01, -1.0296e+00,  1.1646e-01, -6.5707e-01, -5.8304e-01,
         -4.8231e-01, -2.2587e-01,  4.1158e-01, -1.2575e-01,  7.3166e-01,
         -4.0933e-01],
        [-2.7061e-01, -7.7921e-01,  1.4089e-01,  1.3796e-01,  4.1623e-01,
          4.5352e-01, -3.9598e-01,  5.2270e-01, -4.1168e-01, -3.6939e-02,
         -8.0417e-01, -4.1789e-01,  9.5172e-01, -3.0343e-01, -7.5285e-01,
          4.5274e-01, -8.9319e-01, -1.8931e-01, -4.2969e-01, -7.2203e-01,
         -2.6277e-01,  1.1451e-01,  6.3045e-01, -8.4112e-02,  8.3867e-01,
         -2.6915e-01],
        [ 1.9218e-01, -3.8604e-01,  6.3593e-02, -4.6264e-01,  4.9202e-01,
          3.4902e-01, -9.3626e-02,  7.4799e-01, -5.9538e-02, -5.5016e-01,
         -2.8755e-01, -3.2186e-01,  2.5702e-01, -6.6543e-01, -1.09

dict_keys(['pred_logits', 'image', 'backbone_output', 'prepad_input_size', 'neck_output'])

In [15]:
res['pred_logits'].shape

torch.Size([32, 26])

In [4]:
# S.ckpt_path = 'expr_files/vitruvian/checkpoints/TEMP_EXP'
# if not os.path.exists(S.ckpt_path):
#     os.makedirs(S.ckpt_path)
# config_save_to = os.path.join(S.ckpt_path, 'config.yaml')
# if not os.path.exists(config_save_to):
#     shutil.copy(config_path, config_save_to)
# S.initialize(easydict.EasyDict({'expname': 'TEMP_EXP'}))
# S.run()


In [6]:
# del vit_base_patch16_ladder_attention_share_pos_embed
# import importlib
# import path_vit_base
# importlib.reload(path_vit_base)
# from path_vit_base import vit_base_patch16_ladder_attention_share_pos_embed
# backbone_config = C.config['common']['backbone']['kwargs']
# backbone_config['img_size'] = [256, 128]

In [8]:
# import torch.distributed as dist
# import os
# os.environ['MASTER_ADDR'] = 'localhost'
# os.environ['MASTER_PORT'] = '12355'
# os.environ['RANK'] = '0'
# os.environ['WORLD_SIZE'] = '1'
# dist.init_process_group(backend='nccl', init_method='env://')
# pretrained_path = '/dscilab_dungvo/workspace/BA-PRE_THESIS/my_source/OpenGVBackbone/PATH/path_vitbase_size224.pth'
# model = vit_base_patch16_ladder_attention_share_pos_embed(pretrained_path=pretrained_path, **backbone_config)
# model.cpu()
# temp_image = torch.randn(2, 3, 224, 224).cpu()
# res = model(temp_image)
# state_dict = torch.load(pretrained_path, map_location='cpu')

{'pretrain_path': '/mnt/lustre/share/chencheng1/pretrain/L2_final_base/wo_cls_token/v100_32g_vitbase_size224_lr1e3_stepLRx3_bmp1_adafactor_wd01_clip05_layerdecay075_lpe_peddet_citypersons_LSA_reduct8_tbn1_heads2_gate1_peddetShareDecoder_exp3_setting_SharePosEmbed.pth'}


pos embed shape:  torch.Size([1, 128, 768])
[rank 0] Position interpolate from (14, 14) to (16, 8)
Missing keys: []

finish load


PathViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
 